# 01 — Dataset download, validation, and few-shot preparation

**Purpose:** Download the official SIB-200 English/Urdu splits, preserve source IDs, detect leakage, and create repeatable nested training subsets.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# 0. CLONE + ENVIRONMENT SETUP — RUN FIRST (Kaggle/Colab).
# Internet must be ON. The FIRST run may install libraries and STOP for a kernel restart.
# After restarting the kernel, rerun this cell; it will skip installation if versions match.
from pathlib import Path
import os, sys, subprocess, importlib.metadata as metadata

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
PROJECT_FOLDER = "Urdu_Qwen_Research_Notebooks"
PULL_LATEST = False  # Set True after pushing changes to GitHub; requires a clean Git checkout.

if Path('/kaggle/working').is_dir():
    WORKSPACE = Path('/kaggle/working')
elif Path('/content').is_dir():
    WORKSPACE = Path('/content')
else:
    WORKSPACE = Path.cwd()
REPO_DIR = WORKSPACE / 'nlp_research_proj'
ROOT = REPO_DIR / PROJECT_FOLDER

if (REPO_DIR / '.git').is_dir():
    if PULL_LATEST:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    if REPO_DIR.exists():
        raise RuntimeError(f'{REPO_DIR} exists but is not a Git clone. Rename/remove it after backing up outputs.')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
if not (ROOT / 'requirements.txt').is_file() or not (ROOT / 'project_utils.py').is_file():
    raise FileNotFoundError(f'Expected complete project at {ROOT}. Check the GitHub folder layout.')

# This environment was selected for a stable Transformers 4 training API.
if not ((3, 10) <= sys.version_info[:2] <= (3, 12)):
    raise RuntimeError(f'Python {sys.version.split()[0]} not in tested support range 3.10–3.12.')
REQUIRED = {
    'transformers': '4.51.3',
    'peft': '0.15.2',
    'accelerate': '1.6.0',
    'datasets': '3.5.0',
    'tokenizers': '0.21.1',
    'huggingface-hub': '0.30.2',
    'safetensors': '0.5.3',
    'scikit-learn': '1.6.1',
    'pandas': '2.2.3',
    'numpy': '1.26.4',
    'matplotlib': '3.10.1',
    'psutil': '7.0.0',
    'ipywidgets': '8.1.6',
    'nbformat': '5.10.4',
    'pytest': '8.3.5',
}

def installed_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

mismatch = {name: (installed_version(name), version)
            for name, version in REQUIRED.items()
            if installed_version(name) != version}
torchao = installed_version('torchao')
if mismatch or torchao is not None:
    print('Installing the project-locked environment; CUDA-enabled PyTorch is left untouched.')
    if mismatch:
        print('Packages to align:', mismatch)
    if torchao is not None:
        print('Removing optional incompatible torchao:', torchao)
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt'),
                    '--upgrade-strategy', 'only-if-needed'], check=True)
    raise RuntimeError('Installation finished. RESTART THE KAGGLE/COLAB KERNEL, then rerun this FIRST cell. '
                       'Do not train before restarting.')

# No package installation needed: import the shared research code only now.
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()

# Catch future API drift before downloading a 0.5B model.
import inspect
from transformers import TrainingArguments, Trainer
sig = inspect.signature(TrainingArguments).parameters
needed = ['output_dir', 'group_by_length', 'warmup_ratio', 'eval_strategy']
missing = [name for name in needed if name not in sig]
if missing:
    raise RuntimeError(f'Transformers does not match our 4.51.3 training API. Missing: {missing}')
api = check_training_api()
print('Ready. Project:', ROOT)
print('Transformers:', api['transformers'], '| PEFT:', api['peft'])
print('Trainer token argument:', api['trainer_tokenizer_key'])
print('No installation needed; proceed to the next cell.')


**Rules:** Do not alter or resplit the official test data; do not use the paper's 1,000 test headlines for training. Keep Urdu and English labels identical, save source IDs, and verify that no corresponding sentence appears in different splits. Datasets are downloaded from [Davlan/SIB-200](https://huggingface.co/datasets/Davlan/sib200).

In [ ]:
from datasets import load_dataset
from collections import Counter
TASK = 'sib200'
LANGS = {'ur': 'urd_Arab', 'en': 'eng_Latn'}
DATA = {}
for lang, lang_id in LANGS.items():
    DATA[lang] = load_dataset('Davlan/sib200', lang_id)
    print(lang, {s: len(DATA[lang][s]) for s in DATA[lang]})
    print('fields:', DATA[lang]['train'].column_names)
    assert {'index_id', 'category', 'text'} <= set(DATA[lang]['train'].column_names)
    actual_categories = set(DATA[lang]['train']['category'])
    print('Categories:', sorted(actual_categories))
    assert actual_categories == set(DEFAULT_LABELS), f'Unexpected category names: {actual_categories}'


In [ ]:
# SIB-200 uses string `category` values rather than a numeric `label` feature.
labels = list(DEFAULT_LABELS)
label_to_id = {name: index for index, name in enumerate(labels)}
rows = []
for lang, dsd in DATA.items():
    for split in ['train', 'validation', 'test']:
        for r in dsd[split]:
            label = str(r['category'])
            assert label in label_to_id, f'Unexpected label: {label}'
            item = dict(example_id=stable_id(TASK, lang, split, str(r['index_id'])),
                        index_id=str(r['index_id']), source='Davlan/sib200', lang=lang, split=split,
                        text=str(r['text']), label_id=label_to_id[label], label=label)
            rows.append(item)
problems = validate_split(rows, labels)
print('Rows:', len(rows), 'issues:', problems[:10])
assert not problems, 'Inspect split leakage, missing values or duplicate IDs before proceeding.'
for s in ['train', 'validation', 'test']:
    en = {r['index_id']: r['label_id'] for r in rows if r['lang']=='en' and r['split']==s}
    ur = {r['index_id']: r['label_id'] for r in rows if r['lang']=='ur' and r['split']==s}
    assert en == ur, 'Parallel English and Urdu IDs/labels do not match; inspect before continuing.'
print('Parallel languages and labels match by index_id across all splits.')


In [ ]:
out = ROOT/'data'/TASK; out.mkdir(exist_ok=True)
for lang in LANGS:
    for split in ['train','validation','test']:
        group = [r for r in rows if r['lang']==lang and r['split']==split]
        write_csv(out / f'{lang}_{split}.csv', group)
counts = {lang: {s:Counter(r['label'] for r in rows if r['lang']==lang and r['split']==s)
                 for s in ['train','validation','test']} for lang in LANGS}
manifest = {'task':TASK, 'dataset':'Davlan/sib200','language_configs':LANGS, 'labels':labels,
            'rows_by_split': {f'{lang}_{split}':len([r for r in rows if r['lang']==lang and r['split']==split])
                              for lang in LANGS for split in ['train','validation','test']},
            'sha256':{f'{lang}_{split}':file_hash(out/f'{lang}_{split}.csv')
                      for lang in LANGS for split in ['train','validation','test']},
            'notes':'Fixed official splits; class mapping fixed to DEFAULT_LABELS, dataset has string category feature. Do not change test labels.'}
write_json(out/'manifest.json',manifest)
print('Saved:',out)
for lang in LANGS:
    print(lang,'class counts:',{split:dict(c) for split,c in counts[lang].items()})

In [ ]:
for seed in [42,123,456]:
    subsets = stratified_prefix_samples(load_task(TASK,'ur','train'), [5,10,25,50,100], seed=seed)
    for n, entry in subsets.items():
        if entry['available']:
            write_csv(out / f'ur_train_{n}perclass_seed{seed}.csv', entry['rows'])
            print(f'seed={seed} n/class={n} total={len(entry["rows"])}')
        else:
            print(f'seed={seed} n/class={n} NOT available:', entry['shortfall'])
print('Important: 100 examples per class is not guaranteed; do not oversample without explicitly reporting it.')